# Практическая работа 7. AI-агенты и middleware

## Задание 1. Разработка сценария агента

**Сценарий:** учебный помощник, который помогает изучать программирование и машинное обучение.

**Целевая аудитория:** начинающие специалисты, которые хотят изучать технические темы и планировать обучение.

**Логика работы:**
- Агент получает вопрос пользователя и определяет, какой инструмент необходим.
- Может предоставить информацию об учебной теме или рассчитать учебную нагрузку.
- Учитывает уровень подготовки, выбранную тему и предпочтительный стиль ответа.
- Сохраняет историю диалога и использует ранее полученные сведения.
- Перед сохранением учебного плана запрашивает подтверждение пользователя.

**Инструменты:** локальный справочник учебных тем и калькулятор учебной нагрузки.

**Память:** LangGraph InMemorySaver.

Тажке будут добавлены собственное состояние, динамический промпт и human-in-the-loop. Реализация этих механизмов будет выполнена в следующих заданиях.

In [1]:
!pip install -q -U langchain langgraph langchain-gigachat

In [2]:
from google.colab import userdata
from langchain_gigachat import GigaChat

model = GigaChat(
    credentials=userdata.get("GIGACHAT_CREDENTIALS"),
    scope="GIGACHAT_API_PERS",
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0,
)

print("GigaChat initialized")

GigaChat initialized


## Задание 2. Инструменты агента

Создадим два инструмента:

1. `get_topic_info` — получает описание учебной темы из локального справочника.
2. `calculate_study_plan` — рассчитывает количество учебных часов и дней.

Оба инструмента работают локально и не требуют внешних API. Агент будет самостоятельно выбирать нужную функцию в зависимости от запроса пользователя.

In [3]:
from langchain.tools import tool

TOPICS = {
    "python": "Python — язык программирования общего назначения. Подходит для автоматизации, анализа данных и разработки ML-сервисов.",
    "machine learning": "Machine Learning — методы обучения моделей на данных для прогнозирования и принятия решений.",
    "rag": "RAG — подход, при котором LLM получает дополнительную информацию из найденных документов перед генерацией ответа.",
}


@tool
def get_topic_info(topic: str) -> str:
    """Возвращает краткую информацию об учебной теме из локального справочника."""
    return TOPICS.get(
        topic.lower().strip(),
        f"Тема '{topic}' отсутствует в справочнике."
    )


@tool
def calculate_study_plan(total_hours: int, hours_per_day: int) -> str:
    """Рассчитывает количество дней для обучения по общему объёму часов и часам в день."""
    if total_hours <= 0 or hours_per_day <= 0:
        return "Количество часов должно быть положительным."

    days = (total_hours + hours_per_day - 1) // hours_per_day

    return (
        f"Для обучения объёмом {total_hours} часов "
        f"при нагрузке {hours_per_day} ч/день потребуется {days} дней."
    )


tools = [get_topic_info, calculate_study_plan]

print("Tools:", [t.name for t in tools])

Tools: ['get_topic_info', 'calculate_study_plan']


In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "Ты учебный помощник. Отвечай по-русски, кратко и понятно. "
        "Для получения сведений о теме используй get_topic_info. "
        "Для расчёта учебной нагрузки используй calculate_study_plan. "
        "Не придумывай результаты инструментов."
    ),
)

print("Agent created")

Agent created


Проверим вызовы созданных инструментов:

In [5]:
result_1 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Найди в учебном справочнике информацию о Python."
        }
    ]
})

for message in result_1["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : Найди в учебном справочнике информацию о Python.
AIMessage : 
ToolMessage : Python — язык программирования общего назначения. Подходит для автоматизации, анализа данных и разработки ML-сервисов.
AIMessage : Python — это язык программирования общего назначения, подходящий для автоматизации задач, анализа данных и разработки сервисов машинного обучения.


In [6]:
result_2 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Рассчитай, сколько дней потребуется для изучения курса на 30 часов, если заниматься по 2 часа в день."
        }
    ]
})

for message in result_2["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : Рассчитай, сколько дней потребуется для изучения курса на 30 часов, если заниматься по 2 часа в день.
AIMessage : 
ToolMessage : Для обучения объёмом 30 часов при нагрузке 2 ч/день потребуется 15 дней.
AIMessage : Для изучения курса на 30 часов при занятиях по 2 часа в день потребуется 15 дней.


## Задания 3-4. Собственное состояние и динамический промпт

Создадим состояние агента с тремя дополнительными полями:

- `topic` — текущая учебная тема;
- `level` — уровень подготовки пользователя;
- `response_style` — предпочтительный стиль ответа.

Динамический промпт будет использовать эти параметры, чтобы подстраивать объяснения под пользователя. История диалога будет храниться в поле `messages`.

In [7]:
from typing import NotRequired
from langchain.agents import AgentState

class CustomState(AgentState):
    topic: NotRequired[str]
    level: NotRequired[str]
    response_style: NotRequired[str]

print("CustomState created")

CustomState created


Создадим динамический промпт. Используем декоратор @dynamic_prompt из LangChain. Он позволяет формировать системные инструкции перед обращением к модели на основании текущего состояния.

In [8]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def study_dynamic_prompt(request: ModelRequest) -> str:
    state = request.state

    topic = state.get("topic", "не указана")
    level = state.get("level", "beginner")
    style = state.get("response_style", "short")

    if style == "detailed":
        style_instruction = ("Давай подробные объяснения с примерами.")
    else:
        style_instruction = ("Отвечай кратко, максимум в 2-3 предложениях.")

    print(
        f"[Dynamic prompt] topic={topic}, "
        f"level={level}, style={style}"
    )

    return (
        "Ты учебный помощник. Отвечай по-русски.\n"
        f"Текущая тема: {topic}.\n"
        f"Уровень пользователя: {level}.\n"
        f"{style_instruction}\n"
        "Для поиска сведений об учебной теме используй get_topic_info. "
        "Для расчёта учебной нагрузки используй calculate_study_plan."
    )

Добавим локальную память InMemorySaver. Она позволит сохранять сообщения и состояние между обращениями в рамках одного thread_id.

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=tools,
    state_schema=CustomState,
    middleware=[study_dynamic_prompt],
    checkpointer=InMemorySaver()
)

print("Agent with CustomState and Dynamic Prompt created")

Agent with CustomState and Dynamic Prompt created


Проведём два теста с одинаковым вопросом, но разными параметрами состояния.

In [10]:
# начинающий пользователь, краткий ответ

config_short = {"configurable": {"thread_id": "test_short"}}

result_short = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Найди в справочнике информацию о RAG и объясни, что это такое."
            }
        ],
        "topic": "RAG",
        "level": "beginner",
        "response_style": "short",
    },
    config=config_short,
)

print("\nANSWER:")
print(result_short["messages"][-1].content)

[Dynamic prompt] topic=RAG, level=beginner, style=short
[Dynamic prompt] topic=RAG, level=beginner, style=short

ANSWER:
RAG означает Retrieval-Augmented Generation. Это метод, когда модель машинного обучения дополняет свои знания внешними источниками информации перед созданием ответа.


In [11]:
# опытный пользователь, подробный ответ

config_detailed = {"configurable": {"thread_id": "test_detailed"}}

result_detailed = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Найди в справочнике информацию о RAG и объясни, что это такое."
            }
        ],
        "topic": "RAG",
        "level": "advanced",
        "response_style": "detailed",
    },
    config=config_detailed,
)

print("\nANSWER:")
print(result_detailed["messages"][-1].content)

[Dynamic prompt] topic=RAG, level=advanced, style=detailed
[Dynamic prompt] topic=RAG, level=advanced, style=detailed

ANSWER:
RAG (Retrieval-Augmented Generation) – это подход, при котором большие языковые модели (LLM) используют дополнительные источники информации из найденных документов перед тем, как генерировать окончательный ответ. Этот метод позволяет повысить точность и актуальность ответов моделей, поскольку они опираются не только на внутреннюю память, но и на внешние знания, извлечённые из внешних источников.

### Пример работы RAG:

Допустим, пользователь задает вопрос: "Какова средняя температура в Москве в июле?"

1. Модель обращается к внешним источникам (например, поисковым системам или базам знаний) и находит актуальные сведения о погоде в Москве.
2. Затем модель интегрирует эту внешнюю информацию в свой ответ, обеспечивая высокую достоверность и актуальность ответа.
3. Ответ может выглядеть следующим образом:  
   "Средняя температура в Москве в июле составляет около 

Проверим состояние и историю.

In [12]:
for name, config in [("Short", config_short), ("Detailed", config_detailed)]:
    state = agent.get_state(config).values

    print(f"\n{name}:")
    print("Topic:", state.get("topic"))
    print("Level:", state.get("level"))
    print("Style:", state.get("response_style"))
    print("Messages:", len(state["messages"]))
    print("Message types:", [type(m).__name__ for m in state["messages"]])


Short:
Topic: RAG
Level: beginner
Style: short
Messages: 4
Message types: ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']

Detailed:
Topic: RAG
Level: advanced
Style: detailed
Messages: 4
Message types: ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']


## Задание 5. Память агента

Проверим работу памяти `InMemorySaver` на примере расчёта учебной нагрузки.

Сначала пользователь сообщит продолжительность курса и количество часов занятий в день. Затем изменит только ежедневную нагрузку, не повторяя продолжительность курса.

Агент должен восстановить исходные данные из истории диалога и самостоятельно передать их инструменту расчёта.

In [13]:
memory_config = {
    "configurable": {"thread_id": "study_memory_test"}
}

result_1 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Я изучаю Python. Курс рассчитан на 30 часов. "
                "Рассчитай, сколько дней потребуется, "
                "если заниматься по 2 часа в день."
            ),
        }],
        "topic": "Python",
        "level": "beginner",
        "response_style": "short",
    },
    config=memory_config,
)

print("ANSWER:")
print(result_1["messages"][-1].content)

[Dynamic prompt] topic=Python, level=beginner, style=short
[Dynamic prompt] topic=Python, level=beginner, style=short
ANSWER:
При изучении курса длительностью 30 часов с нагрузкой 2 часа в день вам понадобится 15 дней.


In [14]:
# теперь не передаём ни продолжительность курса, ни поля CustomState, просто сообщщаем новую нагрузку

result_2 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "А если я буду заниматься по 3 часа в день, "
                "сколько дней потребуется для того же курса?"
            ),
        }]
    },
    config=memory_config,
)

print("\nANSWER:")
print(result_2["messages"][-1].content)

[Dynamic prompt] topic=Python, level=beginner, style=short
[Dynamic prompt] topic=Python, level=beginner, style=short

ANSWER:
При изучении курса длительностью 30 часов с нагрузкой 3 часа в день вам понадобится 10 дней.


Проверяем вызовы инструментов и состояние:

In [15]:
from langchain_core.messages import AIMessage, ToolMessage

state = agent.get_state(memory_config).values

print("Saved state:")
print("Topic:", state.get("topic"))
print("Level:", state.get("level"))
print("Style:", state.get("response_style"))
print("Messages:", len(state["messages"]))

print("\nTool calls:")

for message in state["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print(f"{call['name']}: {call['args']}")

    elif isinstance(message, ToolMessage):
        print("Tool result:", message.content)

Saved state:
Topic: Python
Level: beginner
Style: short
Messages: 8

Tool calls:
calculate_study_plan: {'hours_per_day': 2, 'total_hours': 30}
Tool result: Для обучения объёмом 30 часов при нагрузке 2 ч/день потребуется 15 дней.
calculate_study_plan: {'hours_per_day': 3, 'total_hours': 30}
Tool result: Для обучения объёмом 30 часов при нагрузке 3 ч/день потребуется 10 дней.


## Задание 6. Human-in-the-loop

Теперь добавим подтверждение человеком перед сохранением учебного плана.

Агент должен предложить действие, приостановиться до подтверждения, а затем либо выполнить инструмент, либо отменить действие.

### Инструмент сохранения плана

Добавим инструмент, который сохраняет учебный план в локальное хранилище. Поскольку сохранение изменяет данные, перед его выполнением агент должен получить подтверждение пользователя.

In [16]:
from langchain.tools import tool

saved_plans = {}

@tool
def save_study_plan(topic: str, total_hours: int, hours_per_day: int) -> str:
    """Сохраняет учебный план с темой, общим количеством часов и нагрузкой в день."""
    saved_plans[topic.lower()] = {"total_hours": total_hours, "hours_per_day": hours_per_day}

    return f"Учебный план по теме '{topic}' сохранён."

Создаём агента с human-in-the-loop:

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl_agent = create_agent(
    model=model,
    tools=[get_topic_info, calculate_study_plan, save_study_plan],
    state_schema=CustomState,
    middleware=[
        study_dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={"save_study_plan": {"allowed_decisions": ["approve", "reject"]}}
        ),
    ],
    checkpointer=InMemorySaver(),
)

print("HITL agent created")

HITL agent created


### Проверка Human-in-the-loop

Попросим агента сохранить учебный план. Ожидаем, что выполнение остановится перед вызовом инструмента и агент запросит разрешение. До подтверждения данные не должны появиться в хранилище.

In [18]:
hitl_config = {"configurable": {"thread_id": "hitl_test_1"}}

hitl_result = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Сохрани мой учебный план: Python, "
                "30 часов всего, по 2 часа в день. "
                "Используй инструмент save_study_plan."
            ),
        }],
        "topic": "Python",
        "level": "beginner",
        "response_style": "short",
    },
    config=hitl_config,
    version="v2",
)

print("Interrupts:", len(hitl_result.interrupts))

for interruption in hitl_result.interrupts:
    print(interruption.value)

print("\nSaved plans:", saved_plans)

[Dynamic prompt] topic=Python, level=beginner, style=short
Interrupts: 1
{'action_requests': [{'name': 'save_study_plan', 'args': {'hours_per_day': 2, 'topic': 'Python', 'total_hours': 30}, 'description': "Tool execution requires approval\n\nTool: save_study_plan\nArgs: {'hours_per_day': 2, 'topic': 'Python', 'total_hours': 30}"}], 'review_configs': [{'action_name': 'save_study_plan', 'allowed_decisions': ['approve', 'reject']}]}

Saved plans: {}


### Подтверждение действия

Передадим решение `approve`. Агент должен продолжить выполнение, вызвать инструмент сохранения и записать учебный план.

In [19]:
from langgraph.types import Command

approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=hitl_config,
    version="v2",
)

print("Remaining interrupts:", len(approved_result.interrupts))
print("Final answer:", approved_result.value["messages"][-1].content)
print("Saved plans:", saved_plans)

[Dynamic prompt] topic=Python, level=beginner, style=short
Remaining interrupts: 0
Final answer: Ваш учебный план по Python сохранён.
Saved plans: {'python': {'total_hours': 30, 'hours_per_day': 2}}


### Отклонение действия

Повторим запрос на сохранение плана, но на этот раз передадим решение `reject`. Агент должен продолжить работу без выполнения инструмента, а данные в хранилище остаться неизменными.

In [20]:
reject_config = {"configurable": {"thread_id": "hitl_test_reject"}}

reject_result = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Сохрани учебный план: Machine Learning, "
                "40 часов всего, по 2 часа в день. "
                "Используй save_study_plan."
            ),
        }],
        "topic": "Machine Learning",
        "level": "beginner",
        "response_style": "short",
    },
    config=reject_config,
    version="v2",
)

print("Interrupts:", len(reject_result.interrupts))

for interruption in reject_result.interrupts:
    print(interruption.value)

print("Saved plans before rejection:", saved_plans)

[Dynamic prompt] topic=Machine Learning, level=beginner, style=short
Interrupts: 1
{'action_requests': [{'name': 'save_study_plan', 'args': {'hours_per_day': 2, 'topic': 'Machine Learning', 'total_hours': 40}, 'description': "Tool execution requires approval\n\nTool: save_study_plan\nArgs: {'hours_per_day': 2, 'topic': 'Machine Learning', 'total_hours': 40}"}], 'review_configs': [{'action_name': 'save_study_plan', 'allowed_decisions': ['approve', 'reject']}]}
Saved plans before rejection: {'python': {'total_hours': 30, 'hours_per_day': 2}}


In [21]:
# отклоняем действие

rejected_result = hitl_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "message": "Пользователь отменил сохранение плана."
        }]
    }),
    config=reject_config,
    version="v2",
)

print("Remaining interrupts:", len(rejected_result.interrupts))
print("Final answer:", rejected_result.value["messages"][-1].content)
print("Saved plans after rejection:", saved_plans)

[Dynamic prompt] topic=Machine Learning, level=beginner, style=short
Remaining interrupts: 0
Final answer: Сохранение учебного плана отменено пользователем.
Saved plans after rejection: {'python': {'total_hours': 30, 'hours_per_day': 2}}


In [22]:
# итоговая проверка

assert saved_plans["python"] == {"total_hours": 30, "hours_per_day": 2}

assert "machine learning" not in saved_plans

print("HITL verification passed")
print("Saved plans:", saved_plans)

HITL verification passed
Saved plans: {'python': {'total_hours': 30, 'hours_per_day': 2}}


## Задание 7. Демонстрация агента

Проверим работу всех механизмов в одном диалоге из пяти запросов.

Агент должен находить информацию через инструменты, помнить параметры курса, изменять стиль ответов в зависимости от состояния и запрашивать подтверждение перед сохранением учебного плана.

In [24]:
# подготовка диалога

from langgraph.types import Command
from langchain_core.messages import AIMessage, ToolMessage

demo_config = {"configurable": {"thread_id": "final_demo"}}

def show_result(result):
    for message in result.value["messages"]:
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                print(f"Tool call: {call['name']}({call['args']})")

    print("Answer:", result.value["messages"][-1].content)

In [25]:
# Сообщение 1. Получение информации из справочника

r1 = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Найди в справочнике информацию о Python."
        }],
        "topic": "Python",
        "level": "beginner",
        "response_style": "short",
    },
    config=demo_config,
    version="v2",
)

print("USER 1: Информация о Python")
show_result(r1)

[Dynamic prompt] topic=Python, level=beginner, style=short
[Dynamic prompt] topic=Python, level=beginner, style=short
USER 1: Информация о Python
Tool call: get_topic_info({'topic': 'Python'})
Answer: Python — это высокоуровневый язык программирования общего назначения, который широко применяется в автоматизации задач, анализе данных и машинном обучении.


In [27]:
# Сообщение 2. Расчёт учебного плана

r2 = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Мой курс Python рассчитан на 30 часов. "
                "Сколько дней нужно при занятиях по 2 часа в день? "
                "Используй калькулятор."
            )
        }]
    },
    config=demo_config,
    version="v2",
)

print("USER 2: Расчёт учебной нагрузки")
show_result(r2)

[Dynamic prompt] topic=Python, level=beginner, style=short
[Dynamic prompt] topic=Python, level=beginner, style=short
USER 2: Расчёт учебной нагрузки
Tool call: get_topic_info({'topic': 'Python'})
Tool call: calculate_study_plan({'hours_per_day': 2, 'total_hours': 30})
Answer: При занятиях по 2 часа в день весь курс Python из 30 часов займёт 15 дней.


In [28]:
# Сообщение 3. Проверка памяти

r3 = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "А если заниматься по 3 часа в день, "
                "сколько займёт тот же курс? Используй калькулятор."
            )
        }]
    },
    config=demo_config,
    version="v2",
)

print("USER 3: Изменение нагрузки")
show_result(r3)

[Dynamic prompt] topic=Python, level=beginner, style=short
[Dynamic prompt] topic=Python, level=beginner, style=short
USER 3: Изменение нагрузки
Tool call: get_topic_info({'topic': 'Python'})
Tool call: calculate_study_plan({'hours_per_day': 2, 'total_hours': 30})
Tool call: calculate_study_plan({'hours_per_day': 3, 'total_hours': 30})
Answer: Курс из 30 часов при занятиях по 3 часа в день завершится за 10 дней.


In [29]:
# Сообщение 4. Изменение динамического промпта

r4 = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Найди в справочнике информацию о RAG "
                "и подробно объясни эту тему."
            )
        }],
        "topic": "RAG",
        "level": "advanced",
        "response_style": "detailed",
    },
    config=demo_config,
    version="v2",
)

print("USER 4: Подробное объяснение RAG")
show_result(r4)

[Dynamic prompt] topic=RAG, level=advanced, style=detailed
[Dynamic prompt] topic=RAG, level=advanced, style=detailed
USER 4: Подробное объяснение RAG
Tool call: get_topic_info({'topic': 'Python'})
Tool call: calculate_study_plan({'hours_per_day': 2, 'total_hours': 30})
Tool call: calculate_study_plan({'hours_per_day': 3, 'total_hours': 30})
Tool call: get_topic_info({'topic': 'RAG'})
Answer: RAG (Retrieval-Augmented Generation) — это метод, при котором модели обработки естественного языка (LLM, Large Language Models) получают возможность дополнять свой ответ информацией из внешних источников, прежде чем генерировать финальный результат. Этот подход позволяет значительно повысить точность и актуальность ответов моделей, поскольку они используют не только внутренние знания, но и внешние источники информации.

### Принцип работы RAG:

1. **Поиск релевантной информации:**  
   Сначала система проводит поиск релевантных фрагментов текста среди заранее подготовленных документов или баз знан

In [30]:
# Сообщение 5. Сохранение плана с HITL

r5 = hitl_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Сохрани мой план по Python: "
                "30 часов всего, по 3 часа в день. "
                "Используй save_study_plan."
            )
        }]
    },
    config=demo_config,
    version="v2",
)

print("USER 5: Сохранение плана")
print("Interrupts:", len(r5.interrupts))

for interruption in r5.interrupts:
    print(interruption.value)

[Dynamic prompt] topic=RAG, level=advanced, style=detailed
USER 5: Сохранение плана
Interrupts: 1
{'action_requests': [{'name': 'save_study_plan', 'args': {'hours_per_day': 3, 'topic': 'Python', 'total_hours': 30}, 'description': "Tool execution requires approval\n\nTool: save_study_plan\nArgs: {'hours_per_day': 3, 'topic': 'Python', 'total_hours': 30}"}], 'review_configs': [{'action_name': 'save_study_plan', 'allowed_decisions': ['approve', 'reject']}]}


In [31]:
# подтверждаем

assert len(r5.interrupts) == 1

r5_approved = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=demo_config,
    version="v2",
)

print("Remaining interrupts:", len(r5_approved.interrupts))
print("Answer:", r5_approved.value["messages"][-1].content)
print("Saved plans:", saved_plans)

[Dynamic prompt] topic=RAG, level=advanced, style=detailed
Remaining interrupts: 0
Answer: Ваш учебный план по Python успешно сохранён. Обучение займёт 10 дней при занятиях по 3 часа в день.
Saved plans: {'python': {'total_hours': 30, 'hours_per_day': 3}}


In [32]:
# финальная проверка

state = hitl_agent.get_state(demo_config).values

print("Topic:", state.get("topic"))
print("Level:", state.get("level"))
print("Style:", state.get("response_style"))
print("Messages:", len(state["messages"]))

assert saved_plans["python"] == {"total_hours": 30, "hours_per_day": 3}
assert len(r5_approved.interrupts) == 0

print("Final demonstration passed")

Topic: RAG
Level: advanced
Style: detailed
Messages: 20
Final demonstration passed


## Общий вывод

В ходе практической работы создан учебный AI-агент на базе LangChain, LangGraph и GigaChat-2.

Реализованы локальные инструменты поиска информации и расчёта учебной нагрузки, собственное состояние `CustomState`, динамический системный промпт и память на основе `InMemorySaver`.

Также настроен Human-in-the-loop: перед сохранением учебного плана агент приостанавливает выполнение и ожидает решения пользователя. Успешно проверены сценарии подтверждения и отклонения действия.

Итоговый диалог из пяти запросов подтвердил, что агент корректно вызывает инструменты, использует информацию из истории, меняет стиль ответов и продолжает работу после подтверждения.